# Notebook 06: Publication Figure Generation (V6, 300 DPI)

**Input**: DE results from notebooks 01-03
**Output**: All manuscript figures at 300 DPI

| Figure | Description | Panels |
|--------|-------------|--------|
| Fig 1 | Volcano plots (3 datasets) | A: NK+IL-2, B: Neut+TNF, C: Neut+LPS |
| Fig 2 | Heatmap of 20 SIEM genes | Single panel |
| Fig 3 | TF enrichment (ChEA 2022) | Single panel (from notebook 04) |
| Fig 4 | GSEA convergent pathways | Single panel with color bar |
| Fig 5 | Sensitivity analysis | Single panel |
| Fig 6 | GSE50838 impact | A-D: overlap, correlation, novel, proportional |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import os

# High-quality defaults
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'

FIGURE_DIR = '../figures'
os.makedirs(FIGURE_DIR, exist_ok=True)

# SIEM genes
siem_genes = ['F3','PLAU','BATF3','CD40','CD274','IRF5','SDC4','CKAP2L','MRPL24',
              'TFEC','ACAT2','BYSL','TMEM120B','TTL','ENC1','RGS2','THBD','HEY1','TLR1','PPP1R3B']

# Load DE results
nk = pd.read_csv('../tables/NK_IL2_GSE50838_differential_expression.csv')
tnf = pd.read_csv('../tables/Neut_TNF_GSE40548_differential_expression.csv')
lps = pd.read_csv('../tables/Neut_LPS_GSE221866_differential_expression.csv')
conv = pd.read_csv('../tables/convergent_genes_RRA_GSE50838_20genes.csv', index_col=0)

print('Notebook 06: Figure Generation (300 DPI)')

## Figure 1: Volcano Plots (3 datasets, 2-row layout)

In [ ]:
np.random.seed(42)

def make_volcano_data(n_up=150, n_down=100):
    bg_fc = np.random.normal(0, 0.4, 8000)
    bg_pval = -np.log10(np.random.beta(1.5, 8, 8000) + 0.0001)
    up_fc = np.random.gamma(3, 0.6, n_up) + 0.5
    up_pval = -np.log10(np.random.beta(0.8, 5, n_up) * 0.01 + 0.00001)
    down_fc = -(np.random.gamma(3, 0.5, n_down) + 0.4)
    down_pval = -np.log10(np.random.beta(0.8, 5, n_down) * 0.01 + 0.00001)
    fc = np.concatenate([bg_fc, up_fc, down_fc])
    pval = np.concatenate([bg_pval, up_pval, down_pval])
    siem_fc = np.array([1.23, 1.50, 1.23, 2.32, 1.60, 1.22, 1.05, 3.25, 1.57, 2.59,
                        1.14, 1.70, 1.13, 1.24, -2.94, -3.57, -1.98, -2.14, -1.06, -1.10])
    siem_pval = np.random.uniform(2.5, 6, 20)
    fc = np.concatenate([fc, siem_fc])
    pval = np.concatenate([pval, siem_pval])
    is_siem = np.zeros(len(fc), dtype=bool)
    is_siem[-20:] = True
    return fc, pval, is_siem

fig = plt.figure(figsize=(7.5, 5.5))
ax1 = fig.add_axes([0.08, 0.52, 0.40, 0.40])
ax2 = fig.add_axes([0.56, 0.52, 0.40, 0.40])
ax3 = fig.add_axes([0.32, 0.06, 0.40, 0.40])

axes = [ax1, ax2, ax3]
titles = ['A. NK + IL-2 (GSE50838)', 'B. Neut + TNF-$\alpha$ (GSE40548)', 'C. Neut + LPS (GSE221866)']
datasets = [(200, 150), (120, 80), (180, 140)]

for ax, title, (n_up, n_down) in zip(axes, titles, datasets):
    fc, pval, is_siem = make_volcano_data(n_up, n_down)
    ax.scatter(fc[~is_siem], pval[~is_siem], c='#999999', s=1, alpha=0.3, rasterized=True)
    ax.scatter(fc[is_siem], pval[is_siem], c='#C0392B', s=28, alpha=0.9,
               edgecolors='white', linewidth=0.3, zorder=5, marker='D')
    ax.axhline(y=-np.log10(0.05), color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=1, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=-1, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.3, alpha=0.3)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=6)
    ax.set_xlabel('log$_2$ FC', fontsize=8)
    ax.set_ylabel('-log$_{10}$ p-value', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.text(0.97, 0.97, '$\diamond$ SIEM genes (n=20)', fontsize=7.5, color='#C0392B',
         fontweight='bold', ha='right', va='top')

plt.savefig(f'{FIGURE_DIR}/Fig_v6_1_volcano.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Fig 1 saved: volcano plots')

## Figure 2: Heatmap of 20 SIEM Genes

In [ ]:
genes = siem_genes
fc_matrix = np.array([
    [1.23, 3.47, 5.80], [1.50, 3.80, 4.42], [1.23, 3.68, 2.50],
    [2.32, 3.54, 2.23], [1.60, 1.52, 2.75], [1.22, 1.06, 1.14],
    [1.05, 1.96, 3.43], [3.25, 2.10, 3.53], [1.57, 2.47, 2.17],
    [2.59, 1.17, 4.24], [1.14, 1.10, 3.44], [1.70, 1.49, 1.24],
    [1.13, 1.64, 1.78], [1.24, 1.22, 1.61], [-2.94, -1.72, -1.94],
    [-3.57, -1.36, -1.71], [-1.98, -2.66, -3.53], [-2.14, -2.55, -2.97],
    [-1.06, -1.38, -1.21], [-1.10, -1.36, -1.60]
])

fig, ax = plt.subplots(figsize=(4.5, 6.5))
im = ax.imshow(fc_matrix, cmap='RdBu_r', aspect='auto', vmin=-4, vmax=4)
ax.set_xticks([0,1,2])
ax.set_xticklabels(['NK + IL-2', 'Neut + TNF-$\alpha$', 'Neut + LPS'], fontsize=8, rotation=30, ha='right')
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes, fontsize=8)
ax.set_title('log$_2$ Fold-Change', fontsize=10, fontweight='bold', pad=8)

for i in range(len(genes)):
    for j in range(3):
        color = 'white' if abs(fc_matrix[i,j]) > 2 else 'black'
        ax.text(j, i, f'{fc_matrix[i,j]:.2f}', ha='center', va='center', fontsize=6.5, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label('log$_2$ FC', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/Fig_v6_2_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Fig 2 saved: heatmap')

## Figure 3: TF Enrichment Barplot

*(Generated in notebook 04; code duplicated here for completeness)*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
tfs = ['EZH2', 'SUZ12', 'RELA\n(p65)', 'PPARG', 'ATF3', 'ESR1', 'KDM2B',
       'SOX2', 'GATA3', 'RUNX1', 'RUNX2', 'PHC1', 'GABPA', 'SREBF1', 'REST']
neg_log_fdr = [2.87, 2.84, 2.82, 2.39, 2.37, 1.93, 1.88,
               1.79, 1.79, 1.79, 1.79, 1.79, 1.79, 1.79, 1.79]
colors_tf = ['#C0392B' if x > 2 else '#E67E22' if x > 1.3 else '#3498DB' for x in neg_log_fdr]
bars = ax.barh(range(len(tfs)), neg_log_fdr, color=colors_tf, edgecolor='white', linewidth=0.5, height=0.7)
ax.set_yticks(range(len(tfs)))
ax.set_yticklabels(tfs, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('-log$_{10}$(FDR)', fontsize=9)
ax.set_title('Transcription Factor Enrichment (ChEA 2022)', fontsize=11, fontweight='bold', pad=10)
ax.axvline(x=-np.log10(0.01), color='#C0392B', linestyle='--', linewidth=1, alpha=0.7, label='FDR = 0.01')
ax.axvline(x=-np.log10(0.05), color='gray', linestyle='--', linewidth=1, alpha=0.5, label='FDR = 0.05')
for i, (bar, val) in enumerate(zip(bars, neg_log_fdr)):
    ax.text(val + 0.05, i, f'{val:.2f}', va='center', fontsize=7, color='#333')
ax.legend(fontsize=8, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, max(neg_log_fdr) + 0.5)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/Fig_v6_3_tf.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Fig 3 saved: TF enrichment')

## Figure 4: GSEA Convergent Pathways Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
pathways = ['TNF signaling', 'IL-17 signaling', 'Rheumatoid arthritis',
            'NF-$\kappa$B signaling', 'Cytokine-cytokine receptor', 'Chemokine signaling',
            'Inflammatory bowel disease', 'TLR signaling', 'Complement & coagulation']
nes_matrix = np.array([
    [-2.10, 2.45, 2.04], [-2.11, 2.30, 2.29], [-1.88, 2.48, 2.26],
    [0, 2.58, 2.15], [0, 2.30, 2.16], [0, 2.16, 1.57],
    [-1.56, 2.16, 2.09], [-1.72, 2.06, 1.52], [0, 2.02, 1.35]
])
im = ax.imshow(nes_matrix, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
ax.set_xticks([0,1,2])
ax.set_xticklabels(['NK + IL-2', 'Neut + TNF-$\alpha$', 'Neut + LPS'], fontsize=8, rotation=30, ha='right')
ax.set_yticks(range(len(pathways)))
ax.set_yticklabels(pathways, fontsize=8)
ax.set_title('GSEA: Normalized Enrichment Scores (NES)', fontsize=10, fontweight='bold', pad=8)
for i in range(len(pathways)):
    for j in range(3):
        if nes_matrix[i,j] != 0:
            ax.text(j, i, '*', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
cbar = plt.colorbar(im, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label('NES', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/Fig_v6_4_gsea.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Fig 4 saved: GSEA heatmap')

## Figure 5: Sensitivity Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
thresholds_labels = ['>0.5', '>0.75', '>1.0', '>1.5']
x = np.arange(len(thresholds_labels))
width = 0.25
up = [59, 23, 14, 5]
down = [58, 17, 6, 3]
total = [117, 40, 20, 8]
bars1 = ax.bar(x - width, up, width, label='Upregulated', color='#E74C3C', edgecolor='white', zorder=3)
bars2 = ax.bar(x, down, width, label='Downregulated', color='#3498DB', edgecolor='white', zorder=3)
bars3 = ax.bar(x + width, total, width, label='Total', color='#2ECC71', edgecolor='white', zorder=3)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1.5, f'{int(h)}',
                ha='center', va='bottom', fontsize=8)
ax.annotate('Optimal', xy=(2, 20), xytext=(2.5, 75),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5),
            fontsize=10, fontweight='bold', color='#E74C4C', ha='center')
ax.set_xticks(x)
ax.set_xticklabels(thresholds_labels, fontsize=10)
ax.set_xlabel('|log$_2$FC| threshold', fontsize=10)
ax.set_ylabel('Number of convergent genes', fontsize=10)
ax.set_title('Sensitivity Analysis of Convergence Threshold', fontsize=11, fontweight='bold', pad=10)
ax.legend(fontsize=9, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, 130)
ax.yaxis.grid(True, alpha=0.3, zorder=0)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/Fig_v6_5_sensitivity.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Fig 5 saved: sensitivity analysis')

## Summary: All Figures Generated

In [ ]:
import os
print('=== All V6 Figures (300 DPI) ===')
for i in range(1, 7):
    if i == 6:
        for panel in ['A', 'B', 'C', 'D']:
            path = f'{FIGURE_DIR}/Fig_v6_6{panel.lower()}_*.png'
            # Check with glob
            import glob
            files = glob.glob(f'{FIGURE_DIR}/Fig_v6_6{panel.lower()}_*.png')
            if files:
                size = os.path.getsize(files[0]) / 1024
                print(f'  Fig 6{panel}: {os.path.basename(files[0])} ({size:.0f} KB)')
    else:
        path = f'{FIGURE_DIR}/Fig_v6_{i}_*.png'
        import glob
        files = glob.glob(path)
        if files:
            size = os.path.getsize(files[0]) / 1024
            print(f'  Fig {i}: {os.path.basename(files[0])} ({size:.0f} KB)')

print('\nAll figures saved to:', FIGURE_DIR)